#### # 1. Data Ingestion & Base Staging Setup

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# Extract data from Bronze staging layer
bronze_df = spark.table("workspace.bronze.erp_px_cat_g1v2")

# Staging step to maintain structural architecture consistency 
standardized_df = bronze_df

#### # 2. Business Rules & Product Categorization Logic

In [0]:
# Pass through columns as-is while appending operational audit tracking timestamp
transformed_df = (
    standardized_df
    # 1. Audit Metadata: Operational tracking timestamp matching SQL's DEFAULT GETDATE()
    .withColumn("dwh_create_date", F.current_timestamp())
)

#### # 3. Final Schema Formatting, Renaming, and Target Storage

In [0]:
# Grouping DDL casting and structural organization together (Ordered Sequence)
final_df = transformed_df.select(
    F.col("id").cast("string"),             # Primary Key / Identifier (Product ID)
    F.col("cat").cast("string"),            # Product Attribute - Category
    F.col("subcat").cast("string"),         # Product Attribute - Subcategory
    F.col("maintenance").cast("string"),    # Operational Attribute - Maintenance Flag
    F.col("dwh_create_date")                # Warehouse Audit Metadata
)

# Reference mapping ordered exactly to match the selection sequence above
RENAME_MAP = {
    "id": "id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance"
}

renamed_df = final_df
for old_name, new_name in RENAME_MAP.items():
    renamed_df = renamed_df.withColumnRenamed(old_name, new_name)

# Write output schema directly to Silver Delta layer (handles truncation automatically via overwrite)
renamed_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.erp_product_category")

# Display organized sample preview rows interactively
renamed_df.limit(10).display()